**Data Quality Detective**

A lightweight data quality profiler in Python

In [3]:

import pandas as pd
from io import StringIO

# Synthetic customer data with intentional quality issues
csv_data = StringIO("""customer_id,age,city,monthly_spend
101,29,Valencia,45.50
102,34,Madrid,60.00
103,,Valencia,32.00
104,150,Sevilla,51.00
105,41,,75.00
106,29,Valencia,45.50
107,27,Madrid,-10.00
108,35,Barcelona,62.00
109,unknown,Valencia,40.00
110,31,Madrid,55.00
""")

df = pd.read_csv(csv_data)

print("Dataset loaded successfully")
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")

display(df)


Dataset loaded successfully
Rows: 10
Columns: 4


,customer_id,age,city,monthly_spend
0,101,29,Valencia,45.5
1,102,34,Madrid,60.0
2,103,NaN,Valencia,32.0
3,104,150,Sevilla,51.0
4,105,41,NaN,75.0
5,106,29,Valencia,45.5
6,107,27,Madrid,-10.0
7,108,35,Barcelona,62.0
8,109,unknown,Valencia,40.0
9,110,31,Madrid,55.0


In [4]:

# Basic data quality checks

print("DATA TYPES")
print(df.dtypes)

print("\nMISSING VALUES")
missing = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(2)
})
display(missing)

print("\nDUPLICATE ROWS")
print(df.duplicated().sum())

print("\nNUMERIC SUMMARY")
display(df.describe())


DATA TYPES
customer_id        int64
age               object
city              object
monthly_spend    float64
dtype: object

MISSING VALUES


,missing_count,missing_pct
customer_id,0,0.0
age,1,10.0
city,1,10.0
monthly_spend,0,0.0



DUPLICATE ROWS
0

NUMERIC SUMMARY


,customer_id,monthly_spend
count,10.00000,10.000000
mean,105.50000,45.600000
std,3.02765,23.011833
min,101.00000,-10.000000
25%,103.25000,41.375000
50%,105.50000,48.250000
75%,107.75000,58.750000
max,110.00000,75.000000


In [5]:

def profile_data(df):
    """Generate a basic data quality profile for a DataFrame."""

    profile = pd.DataFrame({
        "data_type": df.dtypes.astype(str),
        "missing_count": df.isna().sum(),
        "missing_pct": (df.isna().mean() * 100).round(2),
        "unique_values": df.nunique(),
    })

    profile.index.name = "column"
    return profile.reset_index()


quality_profile = profile_data(df)

display(quality_profile)


,column,data_type,missing_count,missing_pct,unique_values
0,customer_id,int64,0,0.0,10
1,age,object,1,10.0,8
2,city,object,1,10.0,4
3,monthly_spend,float64,0,0.0,9


In [6]:

def summarize_data(df):
    """Summarize the dataset dimensions and duplicate rows."""

    summary = {
        "total_rows": len(df),
        "total_columns": len(df.columns),
        "duplicate_rows": int(df.duplicated().sum()),
        "total_missing_cells": int(df.isna().sum().sum()),
    }

    return pd.Series(summary, name="value")


display(summarize_data(df))


,value
total_rows,10
total_columns,4
duplicate_rows,0
total_missing_cells,2


In [7]:

quality_profile.to_csv("quality_profile.csv", index=False)

print("Quality profile exported successfully.")


Quality profile exported successfully.


In [8]:

assert quality_profile["column"].nunique() == len(df.columns)

assert (
    quality_profile["missing_count"].sum()
    == df.isna().sum().sum()
)

assert summarize_data(df)["duplicate_rows"] == df.duplicated().sum()

print("All basic quality checks passed.")


All basic quality checks passed.


In [9]:

def validate_data(df):
    """Identify data quality issues using explicit business rules."""

    issues = []

    # Convert values to numeric for validation without changing the source data
    age = pd.to_numeric(df["age"], errors="coerce")
    spend = pd.to_numeric(df["monthly_spend"], errors="coerce")

    def add_issue(mask, column, rule, severity, message):
        for idx in df.index[mask]:
            issues.append({
                "row_index": idx,
                "customer_id": df.loc[idx, "customer_id"],
                "column": column,
                "rule": rule,
                "severity": severity,
                "value": df.loc[idx, column],
                "message": message,
            })

    # Rule 1: age must be numeric when provided
    invalid_age = df["age"].notna() & age.isna()
    add_issue(
        invalid_age,
        "age",
        "AGE_NOT_NUMERIC",
        "high",
        "Age must be a numeric value."
    )

    # Rule 2: age must be between 0 and 100
    invalid_age_range = age.notna() & ~age.between(0, 100)
    add_issue(
        invalid_age_range,
        "age",
        "AGE_OUT_OF_RANGE",
        "high",
        "Age must be between 0 and 100."
    )

    # Rule 3: monthly spending cannot be negative
    negative_spend = spend.notna() & (spend < 0)
    add_issue(
        negative_spend,
        "monthly_spend",
        "NEGATIVE_SPENDING",
        "medium",
        "Monthly spending cannot be negative."
    )

    columns = [
        "row_index", "customer_id", "column",
        "rule", "severity", "value", "message"
    ]

    return pd.DataFrame(issues, columns=columns)


issues_df = validate_data(df)

display(issues_df)


,row_index,customer_id,column,rule,severity,value,message
0,8,109,age,AGE_NOT_NUMERIC,high,unknown,Age must be a numeric value.
1,3,104,age,AGE_OUT_OF_RANGE,high,150,Age must be between 0 and 100.
2,6,107,monthly_spend,NEGATIVE_SPENDING,medium,-10.0,Monthly spending cannot be negative.


In [10]:

# Expected issues in our synthetic dataset
expected_rules = {
    "AGE_NOT_NUMERIC": 1,
    "AGE_OUT_OF_RANGE": 1,
    "NEGATIVE_SPENDING": 1,
}

actual_rules = issues_df["rule"].value_counts().to_dict()

for rule, expected_count in expected_rules.items():
    actual_count = actual_rules.get(rule, 0)

    assert actual_count == expected_count, (
        f"{rule}: expected {expected_count}, got {actual_count}"
    )

print("All validation rules passed.")
print(f"Total issues detected: {len(issues_df)}")

display(
    issues_df.groupby(["rule", "severity"])
    .size()
    .reset_index(name="issue_count")
)


All validation rules passed.
Total issues detected: 3


,rule,severity,issue_count
0,AGE_NOT_NUMERIC,high,1
1,AGE_OUT_OF_RANGE,high,1
2,NEGATIVE_SPENDING,medium,1


In [11]:

issues_df.to_csv("quality_issues.csv", index=False)

print(f"Exported {len(issues_df)} issues to quality_issues.csv")


Exported 3 issues to quality_issues.csv


In [12]:

def build_quality_report(df, quality_profile, issues_df):
    """Build an executive summary of dataset quality."""

    total_rows = len(df)
    total_cells = df.shape[0] * df.shape[1]

    affected_rows = (
        issues_df["row_index"].nunique()
        if not issues_df.empty
        else 0
    )

    summary = {
        "total_rows": total_rows,
        "total_columns": len(df.columns),
        "total_cells": total_cells,
        "missing_cells": int(df.isna().sum().sum()),
        "duplicate_rows": int(df.duplicated().sum()),
        "validation_issues": len(issues_df),
        "rows_with_validation_issues": affected_rows,
        "rows_with_issues_pct": round(
            affected_rows / total_rows * 100, 2
        ) if total_rows else 0,
    }

    return summary


report = build_quality_report(df, quality_profile, issues_df)

display(pd.DataFrame([report]).T.rename(columns={0: "value"}))


,value
total_rows,10.0
total_columns,4.0
total_cells,40.0
missing_cells,2.0
duplicate_rows,0.0
validation_issues,3.0
rows_with_validation_issues,3.0
rows_with_issues_pct,30.0


In [13]:

def build_quality_report(df, quality_profile, issues_df):
    """Build an executive summary of dataset quality."""

    total_rows = len(df)
    total_cells = df.shape[0] * df.shape[1]

    affected_rows = (
        issues_df["row_index"].nunique()
        if not issues_df.empty
        else 0
    )

    summary = {
        "total_rows": total_rows,
        "total_columns": len(df.columns),
        "total_cells": total_cells,
        "missing_cells": int(df.isna().sum().sum()),
        "duplicate_rows": int(df.duplicated().sum()),
        "validation_issues": len(issues_df),
        "rows_with_validation_issues": affected_rows,
        "rows_with_issues_pct": round(
            affected_rows / total_rows * 100, 2
        ) if total_rows else 0,
    }

    return summary


report = build_quality_report(df, quality_profile, issues_df)

display(pd.DataFrame([report]).T.rename(columns={0: "value"}))


,value
total_rows,10.0
total_columns,4.0
total_cells,40.0
missing_cells,2.0
duplicate_rows,0.0
validation_issues,3.0
rows_with_validation_issues,3.0
rows_with_issues_pct,30.0


In [14]:

def summarize_issues(issues_df):
    """Count issues by validation rule and severity."""

    if issues_df.empty:
        return pd.DataFrame(
            columns=["rule", "severity", "issue_count"]
        )

    return (
        issues_df
        .groupby(["rule", "severity"])
        .size()
        .reset_index(name="issue_count")
        .sort_values("issue_count", ascending=False)
        .reset_index(drop=True)
    )


issue_summary = summarize_issues(issues_df)

display(issue_summary)


,rule,severity,issue_count
0,AGE_NOT_NUMERIC,high,1
1,AGE_OUT_OF_RANGE,high,1
2,NEGATIVE_SPENDING,medium,1


In [15]:

# Export the consolidated report
pd.DataFrame(
    list(report.items()),
    columns=["metric", "value"]
).to_csv("quality_summary.csv", index=False)

quality_profile.to_csv("quality_profile.csv", index=False)
issues_df.to_csv("quality_issues.csv", index=False)
issue_summary.to_csv("quality_issue_summary.csv", index=False)

print("Quality report exported successfully.")

for filename in [
    "quality_summary.csv",
    "quality_profile.csv",
    "quality_issues.csv",
    "quality_issue_summary.csv",
]:
    print(f"- {filename}")


Quality report exported successfully.
- quality_summary.csv
- quality_profile.csv
- quality_issues.csv
- quality_issue_summary.csv


In [16]:

assert report["total_rows"] == len(df)
assert report["total_columns"] == len(df.columns)
assert report["validation_issues"] == len(issues_df)
assert report["rows_with_validation_issues"] <= len(df)

assert len(issue_summary) == issues_df["rule"].nunique()

print("Report validation passed.")


Report validation passed.


In [17]:

%%writefile data_quality.py

import pandas as pd


def profile_data(df):
    """Return data types, missing values and unique counts by column."""
    profile = pd.DataFrame({
        "data_type": df.dtypes.astype(str),
        "missing_count": df.isna().sum(),
        "missing_pct": (df.isna().mean() * 100).round(2),
        "unique_values": df.nunique(),
    })

    profile.index.name = "column"
    return profile.reset_index()


def summarize_data(df):
    """Return general dataset quality metrics."""
    return pd.Series({
        "total_rows": len(df),
        "total_columns": len(df.columns),
        "duplicate_rows": int(df.duplicated().sum()),
        "total_missing_cells": int(df.isna().sum().sum()),
    }, name="value")


def validate_data(df):
    """Detect issues using rules for the example customer dataset."""

    required = {"customer_id", "age", "monthly_spend"}
    missing_columns = required - set(df.columns)

    if missing_columns:
        raise ValueError(
            f"Missing required columns: {sorted(missing_columns)}"
        )

    issues = []

    age = pd.to_numeric(df["age"], errors="coerce")
    spend = pd.to_numeric(df["monthly_spend"], errors="coerce")

    def add_issue(mask, column, rule, severity, message):
        for idx in df.index[mask]:
            issues.append({
                "row_index": idx,
                "customer_id": df.loc[idx, "customer_id"],
                "column": column,
                "rule": rule,
                "severity": severity,
                "value": df.loc[idx, column],
                "message": message,
            })

    add_issue(
        df["age"].notna() & age.isna(),
        "age", "AGE_NOT_NUMERIC", "high",
        "Age must be a numeric value."
    )

    add_issue(
        age.notna() & ~age.between(0, 100),
        "age", "AGE_OUT_OF_RANGE", "high",
        "Age must be between 0 and 100."
    )

    add_issue(
        spend.notna() & (spend < 0),
        "monthly_spend", "NEGATIVE_SPENDING", "medium",
        "Monthly spending cannot be negative."
    )

    columns = [
        "row_index", "customer_id", "column",
        "rule", "severity", "value", "message"
    ]

    return pd.DataFrame(issues, columns=columns)


def build_quality_report(df, issues_df):
    """Build an executive summary of quality metrics."""

    total_rows = len(df)
    total_cells = df.shape[0] * df.shape[1]

    affected_rows = (
        int(issues_df["row_index"].nunique())
        if not issues_df.empty else 0
    )

    return {
        "total_rows": total_rows,
        "total_columns": len(df.columns),
        "total_cells": total_cells,
        "missing_cells": int(df.isna().sum().sum()),
        "duplicate_rows": int(df.duplicated().sum()),
        "validation_issues": len(issues_df),
        "rows_with_validation_issues": affected_rows,
        "rows_with_issues_pct": round(
            affected_rows / total_rows * 100, 2
        ) if total_rows else 0,
    }


def summarize_issues(issues_df):
    """Count issues by validation rule and severity."""

    if issues_df.empty:
        return pd.DataFrame(
            columns=["rule", "severity", "issue_count"]
        )

    return (
        issues_df.groupby(["rule", "severity"])
        .size()
        .reset_index(name="issue_count")
        .sort_values("issue_count", ascending=False)
        .reset_index(drop=True)
    )


def generate_reports(df, output_dir="reports"):
    """Run the analysis and export all report files."""
    from pathlib import Path

    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)

    profile = profile_data(df)
    issues = validate_data(df)
    summary = build_quality_report(df, issues)
    issue_summary = summarize_issues(issues)

    pd.DataFrame(
        list(summary.items()), columns=["metric", "value"]
    ).to_csv(output_path / "quality_summary.csv", index=False)

    profile.to_csv(output_path / "quality_profile.csv", index=False)
    issues.to_csv(output_path / "quality_issues.csv", index=False)
    issue_summary.to_csv(
        output_path / "quality_issue_summary.csv", index=False
    )

    return {
        "summary": summary,
        "profile": profile,
        "issues": issues,
        "issue_summary": issue_summary,
    }


Writing data_quality.py


In [18]:

import importlib
import data_quality as dq

importlib.reload(dq)

results = dq.generate_reports(df)

print("Summary:")
display(pd.DataFrame(
    list(results["summary"].items()),
    columns=["metric", "value"]
))

print("\nDetected issues:")
display(results["issues"])

print("\nIssue summary:")
display(results["issue_summary"])


Summary:


,metric,value
0,total_rows,10.0
1,total_columns,4.0
2,total_cells,40.0
3,missing_cells,2.0
4,duplicate_rows,0.0
5,validation_issues,3.0
6,rows_with_validation_issues,3.0
7,rows_with_issues_pct,30.0



Detected issues:


,row_index,customer_id,column,rule,severity,value,message
0,8,109,age,AGE_NOT_NUMERIC,high,unknown,Age must be a numeric value.
1,3,104,age,AGE_OUT_OF_RANGE,high,150,Age must be between 0 and 100.
2,6,107,monthly_spend,NEGATIVE_SPENDING,medium,-10.0,Monthly spending cannot be negative.



Issue summary:


,rule,severity,issue_count
0,AGE_NOT_NUMERIC,high,1
1,AGE_OUT_OF_RANGE,high,1
2,NEGATIVE_SPENDING,medium,1


In [19]:

from pathlib import Path

Path("data").mkdir(exist_ok=True)
Path("reports").mkdir(exist_ok=True)

# Save the synthetic dataset used in the demonstration
df.to_csv("data/sample_customers.csv", index=False)

# Record the project dependency
Path("requirements.txt").write_text(
    "pandas>=1.5.0\n",
    encoding="utf-8"
)

print("Sample dataset and requirements created.")


Sample dataset and requirements created.


In [20]:

sample_df = pd.read_csv("data/sample_customers.csv")

results_from_file = dq.generate_reports(sample_df)

assert len(results_from_file["issues"]) == 3
assert results_from_file["summary"]["total_rows"] == 10

print("End-to-end test passed.")
print("Input: data/sample_customers.csv")
print("Output: reports/")


End-to-end test passed.
Input: data/sample_customers.csv
Output: reports/


In [21]:

%%writefile README.md

# Data Quality Detective

A lightweight Python tool for profiling datasets, detecting data quality issues, and generating reproducible reports.

## Overview

Data Quality Detective helps analysts identify potential data quality problems before using a dataset for analysis or reporting.

The project uses Python and pandas to calculate quality metrics, apply validation rules, and export structured reports for further inspection.

## Features

- **Data profiling:** data types, missing values, missing-value percentages, and unique values by column.
- **Dataset summary:** row count, column count, missing cells, and exact duplicate rows.
- **Rule-based validation:** detects non-numeric ages, ages outside a configurable example range, and negative monthly spending.
- **Issue reporting:** records the affected row, customer ID, column, rule, severity, and observed value.
- **Report generation:** exports summary metrics, column profiles, validation issues, and issue counts to CSV.
- **Basic validation tests:** checks expected results on a synthetic dataset.

## Tech Stack

- Python
- pandas
- Jupyter Notebook / Google Colab

## Project Structure

```text
data-quality-detective/
├── data/
│   └── sample_customers.csv
├── reports/
│   ├── quality_summary.csv
│   ├── quality_profile.csv
│   ├── quality_issues.csv
│   └── quality_issue_summary.csv
├── data_quality.py
├── data-quality-detective.ipynb
├── requirements.txt
└── README.md
```

## Getting Started

### 1. Install dependencies

```bash
pip install -r requirements.txt
```

### 2. Load the sample dataset

```python
import pandas as pd
import data_quality as dq

df = pd.read_csv("data/sample_customers.csv")
```

### 3. Generate the reports

```python
results = dq.generate_reports(df)

print(results["summary"])
display(results["issues"])
display(results["issue_summary"])
```

The generated CSV reports are saved in the `reports/` directory by default.

## Example Validation Rules

| Rule | Description | Severity |
|---|---|---|
| `AGE_NOT_NUMERIC` | Age is present but cannot be interpreted as a number | High |
| `AGE_OUT_OF_RANGE` | Age falls outside the example range of 0–100 | High |
| `NEGATIVE_SPENDING` | Monthly spending is negative | Medium |

These rules are examples designed for the included synthetic customer dataset. They should be adapted to the business context and data dictionary of each real dataset.

## Output Reports

- `quality_summary.csv`: overall dataset metrics.
- `quality_profile.csv`: data quality metrics for each column.
- `quality_issues.csv`: individual validation issues and affected records.
- `quality_issue_summary.csv`: issue counts by rule and severity.

## Limitations

- The current validation rules are specific to the sample customer dataset.
- Missing values are profiled separately and are not yet included in the validation issue list.
- Duplicate rows are reported as a dataset-level metric rather than individual validation issues.
- The tool flags potential problems; it does not automatically modify or clean the source data.
- The current version accepts an existing pandas DataFrame. Direct CSV ingestion and real-world data sources will be expanded in future iterations.

## Future Improvements

- Support configurable validation rules.
- Add CSV ingestion and schema checks.
- Integrate public data sources and HTML table extraction.
- Improve test coverage and reporting.
- Add visual summaries of data quality metrics.

## Author

Created as a portfolio project to demonstrate practical Python, pandas, data profiling, validation, and reporting skills.


Writing README.md


In [22]:

from pathlib import Path

for path in [
    Path("data_quality.py"),
    Path("README.md"),
    Path("requirements.txt"),
    Path("data/sample_customers.csv"),
    Path("reports/quality_summary.csv"),
    Path("reports/quality_profile.csv"),
    Path("reports/quality_issues.csv"),
    Path("reports/quality_issue_summary.csv"),
]:
    print(f"{'OK' if path.exists() else 'MISSING'}  {path}")


OK  data_quality.py
OK  README.md
OK  requirements.txt
OK  data/sample_customers.csv
OK  reports/quality_summary.csv
OK  reports/quality_profile.csv
OK  reports/quality_issues.csv
OK  reports/quality_issue_summary.csv


In [23]:

import pandas as pd
import data_quality as dq

# Test 1: dataset with no validation issues
valid_df = pd.DataFrame({
    "customer_id": [201, 202],
    "age": [25, 40],
    "city": ["Valencia", "Madrid"],
    "monthly_spend": [30.0, 55.0]
})

valid_issues = dq.validate_data(valid_df)
assert valid_issues.empty
print("Test 1 passed: valid dataset")


# Test 2: empty dataset with the required columns
empty_df = pd.DataFrame(columns=[
    "customer_id", "age", "city", "monthly_spend"
])

empty_issues = dq.validate_data(empty_df)
assert empty_issues.empty

empty_summary = dq.build_quality_report(
    empty_df, empty_issues
)
assert empty_summary["rows_with_issues_pct"] == 0
print("Test 2 passed: empty dataset")


# Test 3: missing required column
incomplete_df = pd.DataFrame({
    "customer_id": [301],
    "age": [35]
})

try:
    dq.validate_data(incomplete_df)
except ValueError:
    print("Test 3 passed: missing required column")
else:
    raise AssertionError(
        "Expected ValueError for missing required columns"
    )


print("All tests passed!")


Test 1 passed: valid dataset
Test 2 passed: empty dataset
Test 3 passed: missing required column
All tests passed!


In [24]:

%%writefile test_data_quality.py

import unittest
import pandas as pd

import data_quality as dq


class TestDataQuality(unittest.TestCase):

    def setUp(self):
        self.valid_df = pd.DataFrame({
            "customer_id": [201, 202],
            "age": [25, 40],
            "city": ["Valencia", "Madrid"],
            "monthly_spend": [30.0, 55.0]
        })

    def test_valid_dataset_has_no_issues(self):
        issues = dq.validate_data(self.valid_df)
        self.assertTrue(issues.empty)

    def test_empty_dataset_has_no_issues(self):
        empty_df = pd.DataFrame(columns=[
            "customer_id", "age", "city", "monthly_spend"
        ])

        issues = dq.validate_data(empty_df)
        self.assertTrue(issues.empty)

        summary = dq.build_quality_report(empty_df, issues)
        self.assertEqual(summary["rows_with_issues_pct"], 0)

    def test_missing_required_column_raises_error(self):
        incomplete_df = pd.DataFrame({
            "customer_id": [301],
            "age": [35]
        })

        with self.assertRaises(ValueError):
            dq.validate_data(incomplete_df)

    def test_detects_expected_validation_issues(self):
        issues = dq.validate_data(self.valid_df)

        self.assertEqual(len(issues), 0)

        problematic_df = pd.DataFrame({
            "customer_id": [401, 402, 403],
            "age": [150, 30, "unknown"],
            "city": ["Valencia", "Madrid", "Sevilla"],
            "monthly_spend": [20.0, -10.0, 35.0]
        })

        issues = dq.validate_data(problematic_df)

        self.assertEqual(len(issues), 3)
        self.assertSetEqual(
            set(issues["rule"]),
            {
                "AGE_OUT_OF_RANGE",
                "NEGATIVE_SPENDING",
                "AGE_NOT_NUMERIC"
            }
        )


if __name__ == "__main__":
    unittest.main()


Writing test_data_quality.py


In [25]:

!python -m unittest -v test_data_quality


test_detects_expected_validation_issues (test_data_quality.TestDataQuality.test_detects_expected_validation_issues) ... ok
test_empty_dataset_has_no_issues (test_data_quality.TestDataQuality.test_empty_dataset_has_no_issues) ... ok
test_missing_required_column_raises_error (test_data_quality.TestDataQuality.test_missing_required_column_raises_error) ... ok
test_valid_dataset_has_no_issues (test_data_quality.TestDataQuality.test_valid_dataset_has_no_issues) ... ok

----------------------------------------------------------------------
Ran 4 tests in 0.014s

OK
